# Tokenizadores (Tokenizers)

Nesta sessão do Colab, exploraremos o mundo dos Tokenizadores.

Você pode executar este notebook em uma CPU gratuita ou localmente na sua máquina, se preferir.


## Lembrete: 2 dicas valiosas para usar o Colab:

**Dica 1:**

Não se preocupe com avisos e mensagens de alerta (warnings)!

**Dica 2:**

No meio da execução do Colab, você pode receber um erro como este:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

Esta é uma mensagem de erro muito enganosa! Por favor, não tente alterar as versões dos pacotes...

Isso acontece porque o Google alternou o ambiente de execução (runtime) do seu Colab, talvez porque o Colab estivesse muito ocupado. A solução é:

1. Menu Ambiente de execução (Runtime) >> Desconectar e excluir ambiente de execução
2. Recarregue o colab do zero e no menu Editar >> Limpar todas as saídas
3. Conecte-se a uma nova T4 usando o botão no canto superior direito
4. Selecione "Ver recursos" no menu superior direito para confirmar que você tem uma GPU
5. Execute novamente as células do colab, de cima para baixo, começando pelas instalações com pip

E tudo deve funcionar perfeitamente - caso contrário, me pergunte!

In [41]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [42]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

# Autenticar no Hugging Face

1. Se ainda não o fez, crie uma conta gratuita no Hugging Face em https://huggingface.co e navegue até Settings (Configurações), depois Create a new API token (Criar novo token de API), concedendo permissões de escrita (write).

**IMPORTANTE**: ao criar sua chave de API do Hugging Face, certifique-se de selecionar permissões de leitura/escrita (read/write) clicando na aba WRITE, caso contrário poderá ter problemas mais tarde.

2. Clique no ícone de "chave" no painel lateral à esquerda do Colab e adicione um novo segredo (secret):
`HF_TOKEN = seu_token`

3. Execute a célula abaixo para fazer login.

In [43]:
# Fazer login no Hugging Face

hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("Chave do HF parece correta até agora")
else:
  print("Chave do HF não foi configurada - por favor clique no ícone de chave no painel esquerdo")
login(hf_token, add_to_git_credential=True)

# Verificar a GPU do Google Colab

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Não conectado a uma GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Sucesso - Conectado a uma Tesla T4")
  else:
    print("NÃO CONECTADO A UMA T4")

Chave do HF parece correta até agora
Thu Jul 30 02:54:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------

# Acessando o Llama 3.1 da Meta

Para usar o fantástico Llama 3.1, a Meta exige que você aceite os termos de serviço deles.

Visite a página do modelo no Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

No topo da página há instruções sobre como concordar com os termos. Se possível, use o mesmo e-mail da sua conta do Hugging Face.

Geralmente a aprovação ocorre em alguns minutos. Assim que for aprovado para qualquer modelo 3.1, a permissão se aplica a toda a família de modelos 3.1. Ocasionalmente a Meta pode não aprovar o acesso imediatamente; se isso acontecer, siga este [guia de solução de problemas](https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8?usp=sharing).

Se a próxima célula retornar um erro, verifique:  
1. Você está autenticado no Hugging Face? Tente executar `login()` para checar se sua chave funciona.
2. Você configurou sua chave de API com permissões completas de leitura e escrita?
3. Se você visitar a página do Llama 3.1 pelo link acima, ela mostra que você tem acesso ao modelo perto do topo?

Também há este notebook de solução de problemas para diagnosticar questões de conectividade com o Hugging Face:  
https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8?usp=sharing


In [44]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)

In [45]:
text = "Estou animado para testar os Tokenizadores"
tokens = tokenizer.encode(text)
tokens

[128000, 14101, 283, 4039, 2172, 3429, 1296, 277, 2709, 9857, 450, 18745]

In [46]:
character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f"Existem {character_count} caracteres, {word_count} palavras e {token_count} tokens")

Existem 42 caracteres, 6 palavras e 12 tokens


In [47]:
tokenizer.decode(tokens)

'<|begin_of_text|>Estou animado para testar os Tokenizadores'

In [48]:
tokenizer.batch_decode(tokens)

['<|begin_of_text|>',
 'Est',
 'ou',
 ' anim',
 'ado',
 ' para',
 ' test',
 'ar',
 ' os',
 ' Token',
 'iz',
 'adores']

In [49]:
# tokenizer.vocab
tokenizer.get_added_vocab()

{'<|begin_of_text|>': 128000,
 '<|end_of_text|>': 128001,
 '<|reserved_special_token_0|>': 128002,
 '<|reserved_special_token_1|>': 128003,
 '<|finetune_right_pad_id|>': 128004,
 '<|reserved_special_token_2|>': 128005,
 '<|start_header_id|>': 128006,
 '<|end_header_id|>': 128007,
 '<|eom_id|>': 128008,
 '<|eot_id|>': 128009,
 '<|python_tag|>': 128010,
 '<|reserved_special_token_3|>': 128011,
 '<|reserved_special_token_4|>': 128012,
 '<|reserved_special_token_5|>': 128013,
 '<|reserved_special_token_6|>': 128014,
 '<|reserved_special_token_7|>': 128015,
 '<|reserved_special_token_8|>': 128016,
 '<|reserved_special_token_9|>': 128017,
 '<|reserved_special_token_10|>': 128018,
 '<|reserved_special_token_11|>': 128019,
 '<|reserved_special_token_12|>': 128020,
 '<|reserved_special_token_13|>': 128021,
 '<|reserved_special_token_14|>': 128022,
 '<|reserved_special_token_15|>': 128023,
 '<|reserved_special_token_16|>': 128024,
 '<|reserved_special_token_17|>': 128025,
 '<|reserved_special_to

In [50]:
len(tokenizer.vocab)

128256

# Variantes Instruct dos modelos

Muitos modelos possuem uma variante treinada especificamente para uso em chat.  
Essas variantes geralmente contêm a palavra "Instruct" ao final do nome.  
Eles foram treinados para esperar prompts em um formato específico contendo mensagens de sistema (system), usuário (user) e assistente (assistant).  

Existe o método utilitário `apply_chat_template` que converte a lista de mensagens no formato padrão para o prompt de entrada correto exigido pelo modelo.

In [51]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

In [52]:

messages = [
    {"role": "system", "content": "Você é um assistente prestativo"},
    {"role": "user", "content": "Conte uma piada leve para uma sala cheia de Cientistas de Dados"}
  ]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Você é um assistente prestativo<|eot_id|><|start_header_id|>user<|end_header_id|>

Conte uma piada leve para uma sala cheia de Cientistas de Dados<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## O Momento de Descoberta

Até agora, pode ter parecido que as LLMs recebem diretamente uma lista de dicionários Python como este:

```python
messages = [
    {"role": "system", "content": "Você é um assistente prestativo"},
    {"role": "user", "content": "Conte uma piada leve para uma sala cheia de Cientistas de Dados"}
  ]
```

Porém, uma LLM é apenas um modelo estatístico de Ciência de Dados que recebe uma sequência de números e prevê a probabilidade do próximo número! Não é possível passar objetos Python diretamente para um modelo estatístico.

### E agora você tem a peça que faltava no quebra-cabeça:

As mensagens no formato estilo OpenAI são convertidas:

1. ...em uma sequência de texto com tags especiais para separar as instruções de Sistema, Usuário e Assistente
2. ...em seguida, as palavras são divididas em fragmentos chamados "tokens"
3. ...depois, os tokens são substituídos por IDs de Token (Token IDs) - e essa é a sequência numérica de entrada

> A entrada de uma LLM é uma sequência de IDs de Tokens. A saída é a distribuição de probabilidade do próximo ID de Token após essa entrada.

É isso!


# Testando novos modelos

Agora trabalharemos com 3 modelos:

Phi4 da Microsoft  
DeepSeek 3.1 da DeepSeek AI  
QwenCoder 2.5 da Alibaba Cloud

In [53]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [54]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = "Estou curioso e animado para mostrar os Tokenizadores da Hugging Face em ação para meus engenheiros de LLM"
print("Llama:")
tokens = tokenizer.encode(text)
print(tokens)
print(tokenizer.batch_decode(tokens))
print("\nPhi 4:")
tokens = phi4_tokenizer.encode(text)
print(tokens)
print(phi4_tokenizer.batch_decode(tokens))


Llama:
[128000, 14101, 283, 2917, 59548, 384, 4039, 2172, 3429, 44108, 2709, 9857, 450, 18745, 3067, 473, 36368, 19109, 991, 264, 6027, 3429, 757, 355, 2995, 268, 383, 48328, 409, 445, 11237]
['<|begin_of_text|>', 'Est', 'ou', ' cur', 'ioso', ' e', ' anim', 'ado', ' para', ' mostrar', ' os', ' Token', 'iz', 'adores', ' da', ' H', 'ugging', ' Face', ' em', ' a', 'ção', ' para', ' me', 'us', ' eng', 'en', 'he', 'iros', ' de', ' L', 'LM']

Phi 4:
[152891, 138653, 319, 5727, 1064, 1209, 35989, 1994, 17951, 170058, 1033, 59116, 4512, 29049, 863, 44519, 1209, 47652, 78894, 69590, 334, 451, 19641]
['Estou', ' curioso', ' e', ' anim', 'ado', ' para', ' mostrar', ' os', ' Token', 'izadores', ' da', ' Hug', 'ging', ' Face', ' em', ' ação', ' para', ' meus', ' engen', 'heiros', ' de', ' L', 'LM']


In [55]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Você é um assistente prestativo<|eot_id|><|start_header_id|>user<|end_header_id|>

Conte uma piada leve para uma sala cheia de Cientistas de Dados<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Phi 4:
<|system|>Você é um assistente prestativo<|end|><|user|>Conte uma piada leve para uma sala cheia de Cientistas de Dados<|end|><|assistant|>


In [56]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "Estou curioso e animado para mostrar os Tokenizadores da Hugging Face em ação para meus engenheiros de LLM"
print(tokenizer.encode(text))
print()
print(phi4_tokenizer.encode(text))
print()
print(deepseek_tokenizer.encode(text))

[128000, 14101, 283, 2917, 59548, 384, 4039, 2172, 3429, 44108, 2709, 9857, 450, 18745, 3067, 473, 36368, 19109, 991, 264, 6027, 3429, 757, 355, 2995, 268, 383, 48328, 409, 445, 11237]

[152891, 138653, 319, 5727, 1064, 1209, 35989, 1994, 17951, 170058, 1033, 59116, 4512, 29049, 863, 44519, 1209, 47652, 78894, 69590, 334, 451, 19641]

[0, 15167, 293, 1633, 52599, 312, 5956, 2807, 3583, 88116, 5688, 47948, 571, 18984, 2945, 24133, 5426, 11906, 980, 103464, 3583, 678, 349, 576, 2536, 263, 43187, 392, 33792, 47]


In [57]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Você é um assistente prestativo<|eot_id|><|start_header_id|>user<|end_header_id|>

Conte uma piada leve para uma sala cheia de Cientistas de Dados<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Phi:
<|system|>Você é um assistente prestativo<|end|><|user|>Conte uma piada leve para uma sala cheia de Cientistas de Dados<|end|><|assistant|>

DeepSeek:
<｜begin▁of▁sentence｜>Você é um assistente prestativo<｜User｜>Conte uma piada leve para uma sala cheia de Cientistas de Dados<｜Assistant｜></think>


In [58]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def ola_mundo(pessoa):
  print("Olá", pessoa)
"""
tokens = qwen_tokenizer.encode(code)
for token in tokens:
  print(f"{token}={qwen_tokenizer.decode(token)}")

198=

750=def
297= o
4260=la
717=_m
13499=undo
1295=(p
27587=essoa
982=):

220= 
1173= print
445=("
42719=Ol
1953=á
497=",
56630= pessoa
340=)

